In [35]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

load_dotenv()

True

In [36]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [37]:
from langgraph.graph.message import add_messages 

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] # All types of messages can be included.



In [38]:
def chat_node(state: ChatState) -> ChatState:
    
    messages = state['messages']

    response = model.invoke(messages)

    return {
        'messages': [response]
    }

In [39]:
graph = StateGraph(ChatState)
checkpointer = MemorySaver()

# nodes
graph.add_node('chat_node', chat_node)

# edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chat_bot = graph.compile(checkpointer=checkpointer)

In [40]:
initial_state = {
    'messages': [HumanMessage(content="Hello, how are you?")]
}

In [41]:
# Without Config

# while True:
#     user_input = input("You: ")
#     if user_input.strip().lower() in ['exit', 'quit']:
#         break

#     initial_state['messages'].append(HumanMessage(content=user_input))
#     response_state = chat_bot.invoke(initial_state)
#     response_message = response_state['messages'][-1]  # Get the latest response
#     initial_state['messages'].append(AIMessage(content=response_message.content))  # Add the response to the conversation history
#     print(f"You: {user_input}")
#     print(f"Bot: {response_message.content}")



In [42]:
thread_id = '1'

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        break

    print(f"User : {user_input}")

    config = {
        'configurable': {
            'thread_id': thread_id
        }
    }
    response = chat_bot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)
    
    print(f"AI: {response['messages'][-1].content}")

User : hi
AI: Hello! 👋 How can I help you today?
User : I'm AK
AI: Nice to meet you, AK! 👋 How can I assist you today?
User : Tell me my name
AI: Your name is AK. 😊 Let me know if there’s anything else you’d like to chat about!


In [43]:
chat_bot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='a9280f28-b568-431f-ae02-366126beaaa0'), AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 68, 'total_tokens': 104}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019de79f-10be-7e52-8272-1fdb5ef4b1fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 68, 'output_tokens': 36, 'total_tokens': 104}), HumanMessage(content="I'm AK", additional_kwargs={}, response_metadata={}, id='ab8a521d-8dbb-4f75-96c4-b253c02e55fa'), AIMessage(content='Nice to meet you, AK! 👋 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 91, 'total_tokens': 147}, 'model_name': 'openai/gpt-oss-120b', 'system_finger